## 0. Setup — Otomatis ambil dataset dari Google Drive

Notebook ini akan mencoba membaca dataset dari:
`/content/drive/MyDrive/ultra-milk-yolo-ready/`

Kalau tidak ditemukan, kamu akan diminta mengupload `ultra-milk-yolo-ready.zip`.


In [ ]:
# ============================================
# Langkah 1: Mount Google Drive dan cari dataset
# ============================================
from google.colab import drive
from google.colab import files
from pathlib import Path
import zipfile

# Mount Drive (akan muncul link autentikasi)
drive.mount('/content/drive')

DRIVE_PATH = Path('/content/drive/MyDrive/ultra-milk-yolo-ready')
CONTENT_PATH = Path('/content/ultra-milk-yolo-ready')
ZIP_PATH = Path('/content/drive/MyDrive/ultra-milk-yolo-ready.zip')

def ensure_dataset(base: Path) -> Path:
    """Verifikasi data.yaml ada di base path."""
    if base.exists() and (base / 'data.yaml').exists():
        return base
    return None

# Prioritas 1: dataset sudah ada di Drive
if DRIVE_PATH.exists() and (DRIVE_PATH / 'data.yaml').exists():
    base = DRIVE_PATH
    print(f"Dataset ditemukan di Google Drive: {base}")
# Prioritas 2: zip ada di Drive, ekstrak ke /content/
elif ZIP_PATH.exists():
    print(f"Menemukan zip di Drive: {ZIP_PATH}")
    print("Mengekstrak ke /content/...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    base = ensure_dataset(CONTENT_PATH)
    if base is None:
        raise FileNotFoundError(
            f"Zip diekstrak, tetapi data.yaml tidak ada di {CONTENT_PATH}."
        )
    print(f"Dataset siap di: {base}")
# Prioritas 3: sudah ada di /content/ dari unzip manual
elif CONTENT_PATH.exists() and (CONTENT_PATH / 'data.yaml').exists():
    base = CONTENT_PATH
    print(f"Dataset siap di Colab: {base}")
# Prioritas 4: fallback — minta upload zip
else:
    print("Dataset tidak ditemukan di Drive maupun /content/.")
    print("Silakan upload ultra-milk-yolo-ready.zip di dialog berikut: \n")
    uploaded = files.upload()
    
    zip_name = next(iter(uploaded.keys()))
    zip_path = Path('/content') / zip_name
    
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('/content/')
    
    base = ensure_dataset(CONTENT_PATH)
    if base is None:
        raise FileNotFoundError(
            f"Zip diupload, tetapi data.yaml tidak ditemukan di {CONTENT_PATH}. "
            "Pastikan file zip valid."
        )
    print(f"Dataset siap di: {base}")

# Set global paths untuk cell berikutnya
DATA_BASE = str(base)
DATA_YAML = str(base / 'data.yaml')

print("\nKonfigurasi dataset:")
print(f"  Base   : {DATA_BASE}")
print(f"  YAML   : {DATA_YAML}")


## 1. Install Ultralytics + Cek GPU

In [ ]:
!pip install -q ultralytics

import torch
print('PyTorch version    :', torch.__version__)
print('CUDA available     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU                :', torch.cuda.get_device_name(0))


## 2. Train Model YOLOv8

In [ ]:
from ultralytics import YOLO

use_cuda = torch.cuda.is_available()

print('Data yaml  :', DATA_YAML)
print('Epochs     :', 10)
print('Device     :', 'cuda' if use_cuda else 'cpu')

model = YOLO('yolov8n.pt')
model.train(
    data=DATA_YAML,
    epochs=10,
    imgsz=640,
    batch=16,
    device='cuda' if use_cuda else 'cpu',
    workers=2,
    cache=False,
    project='/content/runs/detect',
    name='train',
    exist_ok=True,
)


## 3. Download best.pt

In [ ]:
from google.colab import files
from pathlib import Path

best = Path('/content/runs/detect/train/weights/best.pt')
if not best.exists():
    raise FileNotFoundError(f'best.pt tidak ditemukan di {best}. Pastikan training selesai.')

files.download(str(best))
print('Downloaded:', best)
